# Mega Project 2 — Regulatory Capital & Expected Loss
## Problem 5: Capital Concentration by Segment — Real Herfindahl-Hirschman Index
## Across Every Real Segment/Geography Dimension This Dataset Provides

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Notebook 01's Basel closed-form capital charge is built on the
infinite-granularity (ASRF) assumption — it assumes idiosyncratic/name
concentration risk is fully diversified away and charges ZERO capital for
it by construction ([BCBS05]). Basel's own Pillar 2 framework requires
banks to separately assess concentration risk that Pillar 1 does not
price. This notebook is that separate assessment: a real
Herfindahl-Hirschman Index (HHI) of capital concentration across every
real segment/geography dimension this dataset actually provides.

### This notebook trains no model and introduces no new baseline assumption
It reuses Notebook 01's already-computed, already-disclosed real
per-applicant capital output unchanged (hard dependency — fails loudly if
Notebook 01 has not been run yet), joined with real application-level
segment columns already used by Notebook 02 (income type, education,
contract type, region rating).

### Why HHI, and why this is a genuine, not redundant, addition to Problem 1
HHI = sum of squared capital shares per segment — the standard
concentration metric borrowed from competition economics (U.S. DOJ/FTC
Horizontal Merger Guidelines interpretive bands: HHI < 1,500
unconcentrated, 1,500–2,500 moderately concentrated, > 2,500 highly
concentrated, on the conventional 0–10,000-point scale), widely used in
credit-portfolio concentration-risk practice. This is a disclosed,
borrowed interpretive convention, not a Basel-mandated portfolio
threshold — stated plainly, not implied.

### Advanced error tackling applied (see LESSONS_LEARNED.md for the
### incidents each of these prevents a repeat of)
- Hard dependency checked by actual required COLUMNS present, not just
  file existence (LESSONS_LEARNED.md #4).
- Real cross-check #1: every dimension's segment capital totals sum
  EXACTLY back to the real portfolio total (catches a join/groupby bug
  immediately rather than silently under- or over-counting capital)
  (LESSONS_LEARNED.md #6).
- Real cross-check #2: HHI is mathematically bounded in [1/N, 1] for a
  dimension with N segments — checked directly, not assumed.
- No `monotonic_within_noise()` risk in this notebook at all —
  concentration analysis has no expected "ordering" across unordered
  categorical segments, so the directionality-convention bug that hit
  Notebooks 01/02 does not apply here by construction
  (LESSONS_LEARNED.md #2).

### Swift processing
One vectorized `pandas` groupby-aggregate per real dimension (the same
pattern already proven fast in Notebook 02 — 1.2 seconds on the user's
real 307,511-applicant portfolio), never a per-applicant Python loop.

### Concentration Validation Verdict vs. Pipeline Integrity Checks
Deliberately NOT named "Statistical Robustness Verdict" like the
TARGET-based notebooks elsewhere in this suite: HHI is a deterministic
concentration measure, not a statistical test against a real TARGET. This
tier instead validates the computation is mathematically correct and
internally consistent (bounds, sum-to-total identity), stated explicitly
rather than forcing an ill-fitting statistical-significance test onto a
deterministic metric.

### Verification status
Verified end-to-end on the synthetic fixture via real Jupyter execution — 0
errors, all integrity and concentration-validation checks pass, HTML
dashboard confirmed under a network-blocked Playwright check, Excel
workbook confirmed via LibreOffice headless recalculation. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 05 — MEGA PROJECT 2: REGULATORY CAPITAL & EXPECTED LOSS
# PROBLEM 5: CAPITAL CONCENTRATION BY SEGMENT
# Real per-applicant capital output (Notebook 01, reused not recomputed),
# analyzed for concentration via the Herfindahl-Hirschman Index (HHI) across
# every real segment/geography dimension this dataset actually has.
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains no model and introduces
# NO new PD/LGD/EAD/correlation value -- it reuses Notebook 01's already-
# computed, already-disclosed real per-applicant capital output
# (decision_engine/artifacts/notebook_01_capital_scores.csv) unchanged, and
# real application-level segment columns already used by Notebook 02 (income
# type, education, contract type, region rating -- the real geography proxy
# this dataset provides). Hard dependency: fails loudly and immediately if
# Notebook 01 has not been run yet.
#
# WHY HHI, AND WHY THIS IS A GENUINE, NOT REDUNDANT, ADDITION TO PROBLEM 1:
# Notebook 01's Basel closed-form capital charge is built on the
# infinite-granularity (ASRF) assumption -- it assumes idiosyncratic/name
# concentration risk is fully diversified away and charges ZERO capital for
# it by construction ([BCBS05]). Basel's own Pillar 2 framework requires
# banks to separately assess concentration risk that Pillar 1 does not
# price -- this notebook is that separate assessment, using the
# Herfindahl-Hirschman Index (HHI = sum of squared capital shares), the
# standard concentration metric borrowed from competition economics
# (U.S. DOJ/FTC Horizontal Merger Guidelines interpretive bands: HHI < 1,500
# unconcentrated, 1,500-2,500 moderately concentrated, > 2,500 highly
# concentrated, on the conventional 0-10,000-point scale) and widely used in
# credit-portfolio concentration-risk practice. This is a disclosed,
# borrowed interpretive convention, not a Basel-mandated portfolio
# threshold -- stated plainly, not implied.
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (see
# LESSONS_LEARNED.md -- every item below cites which real incident it
# prevents a repeat of):
#   - HARD DEPENDENCY compares actual required COLUMNS present, not just
#     file existence (LESSONS_LEARNED.md #4).
#   - REAL CROSS-CHECKS, not asserted: (1) HHI is mathematically bounded in
#     [1/N, 1] for a dimension with N segments -- checked directly, not
#     assumed; (2) every dimension's segment capital totals are checked to
#     sum EXACTLY back to the real portfolio total (catches a join/groupby
#     bug immediately rather than silently under- or over-counting
#     capital) (LESSONS_LEARNED.md #6).
#   - SWIFT, VECTORIZED PROCESSING: one `pandas`/`polars` groupby-aggregate
#     per real dimension (the same pattern already proven fast in Notebook
#     02 -- 1.2 seconds on the user's real 307,511-applicant portfolio),
#     never a per-applicant Python loop.
#   - NO monotonic_within_noise() risk in this notebook -- concentration
#     analysis has no expected "ordering" across unordered categorical
#     segments, so no directionality-convention risk exists here at all
#     (LESSONS_LEARNED.md #2 does not apply by construction).
#   - WARP hardware fix before any heavy import; RAM-headroom checks;
#     HYPER reuse (report_builder); never git operations via the
#     device-mounted folder.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP2_ARTIFACTS_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "artifacts"
ARTIFACTS_DIR = MP2_ARTIFACTS_DIR
REPORTS_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import)
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

T0 = time.time()

from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load Notebook 01's real per-applicant output (HARD dependency,
# checked by actual required columns, not just file existence -- LESSONS
# LEARNED.md #4).
# ---------------------------------------------------------------------------
NB01_SCORES_PATH = ARTIFACTS_DIR / "notebook_01_capital_scores.csv"
if not NB01_SCORES_PATH.exists():
    raise FileNotFoundError(
        "Mega Project 2 / Notebook 05 requires Mega Project 2 / Notebook 01's real "
        "per-applicant output, which has not been produced on this machine yet. Fix: run "
        "02_mega_project_2_regulatory_capital/notebooks/01_expected_loss_capital_requirement.ipynb "
        "end-to-end first, then re-run this notebook."
    )
scores = pl.read_csv(NB01_SCORES_PATH)
N_SCOPE = scores.height
print(f"[LOAD] Real per-applicant capital output from Notebook 01: {N_SCOPE:,} rows.")

required_cols = ["SK_ID_CURR", "CAPITAL_SEGMENT", "CAPITAL_REQUIREMENT", "EAD_PROXY"]
missing_cols = [c for c in required_cols if c not in scores.columns]
if missing_cols:
    raise KeyError(
        f"Required columns missing from notebook_01_capital_scores.csv: {missing_cols}. "
        "Re-run Notebook 01 (it may be an older output format)."
    )

app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)

SEGMENT_DIMENSIONS = ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_CONTRACT_TYPE", "REGION_RATING_CLIENT"]
missing_dims = [c for c in SEGMENT_DIMENSIONS if c not in app.columns]
if missing_dims:
    raise KeyError(f"Required real segment columns missing from application_train.csv: {missing_dims}")

df = scores.join(app.select(["SK_ID_CURR"] + SEGMENT_DIMENSIONS), on="SK_ID_CURR", how="left")
df = df.with_columns(pl.col("REGION_RATING_CLIENT").cast(pl.Utf8))
pdf = df.to_pandas()

ALL_DIMENSIONS = ["CAPITAL_SEGMENT"] + SEGMENT_DIMENSIONS  # CAPITAL_SEGMENT needs no join -- already in Notebook 01's output
REAL_TOTAL_CAPITAL = float(pdf["CAPITAL_REQUIREMENT"].sum())
print(f"[JOIN] Joined real application-level segment columns onto Notebook 01's real capital output. "
      f"{len(ALL_DIMENSIONS)} real dimensions in scope: {ALL_DIMENSIONS}.")

# ---------------------------------------------------------------------------
# SECTION 5 — Real, vectorized HHI computation per dimension (swift: one
# groupby-aggregate per dimension, never a per-applicant loop -- the same
# pattern already proven fast in Notebook 02).
# ---------------------------------------------------------------------------
DOJ_FTC_UNCONCENTRATED_THRESHOLD = 1500.0   # 0-10,000-point scale, borrowed convention (see module docstring)
DOJ_FTC_HIGHLY_CONCENTRATED_THRESHOLD = 2500.0


def _interpret_hhi(hhi_points: float) -> str:
    if hhi_points < DOJ_FTC_UNCONCENTRATED_THRESHOLD:
        return "Unconcentrated"
    if hhi_points < DOJ_FTC_HIGHLY_CONCENTRATED_THRESHOLD:
        return "Moderately Concentrated"
    return "Highly Concentrated"


def _hhi_for_dimension(dim: str) -> tuple[dict, pd.DataFrame]:
    seg = (
        pdf.groupby(dim, observed=True)
        .agg(n_applicants=("SK_ID_CURR", "size"), total_capital=("CAPITAL_REQUIREMENT", "sum"))
        .reset_index()
        .rename(columns={dim: "segment_value"})
    )
    seg["dimension"] = dim
    dim_total_capital = float(seg["total_capital"].sum())
    seg["capital_share"] = seg["total_capital"] / dim_total_capital if dim_total_capital > 0 else 0.0
    n_segments = len(seg)
    hhi_fraction = float((seg["capital_share"] ** 2).sum())
    hhi_points = hhi_fraction * 10_000.0
    effective_n = 1.0 / hhi_fraction if hhi_fraction > 0 else float("nan")
    summary = {
        "dimension": dim, "n_segments": n_segments, "total_capital": dim_total_capital,
        "hhi_fraction": hhi_fraction, "hhi_points": hhi_points, "effective_n_segments": effective_n,
        "interpretation": _interpret_hhi(hhi_points),
        "largest_segment": str(seg.loc[seg["total_capital"].idxmax(), "segment_value"]),
        "largest_segment_share": float(seg["capital_share"].max()),
    }
    return summary, seg.sort_values("total_capital", ascending=False).reset_index(drop=True)


hhi_summaries = []
segment_tables = {}
for dim in ALL_DIMENSIONS:
    summary, seg_table = _hhi_for_dimension(dim)
    hhi_summaries.append(summary)
    segment_tables[dim] = seg_table
    print(f"[HHI] {dim}: {summary['n_segments']} real segments, HHI={summary['hhi_points']:.0f} "
          f"({summary['interpretation']}), effective N={summary['effective_n_segments']:.2f}, "
          f"largest segment '{summary['largest_segment']}' ({summary['largest_segment_share']:.1%} of "
          f"real capital).")
hhi_df = pd.DataFrame(hhi_summaries)
all_segments_df = pd.concat(segment_tables.values(), ignore_index=True)
HHI_RUNTIME_S = round(time.time() - T0, 2)

# ---------------------------------------------------------------------------
# SECTION 6 — Real cross-checks (not asserted -- LESSONS_LEARNED.md #6):
# (1) every dimension's segment capital sums EXACTLY back to the real
#     portfolio total; (2) HHI is mathematically bounded in [1/N, 1] for a
#     dimension with N segments (perfect equal split = 1/N, single-segment
#     concentration = 1).
# ---------------------------------------------------------------------------
_sum_check_rows = []
for dim in ALL_DIMENSIONS:
    dim_total = float(segment_tables[dim]["total_capital"].sum())
    rel_diff = abs(dim_total - REAL_TOTAL_CAPITAL) / REAL_TOTAL_CAPITAL if REAL_TOTAL_CAPITAL > 0 else 0.0
    _sum_check_rows.append({"dimension": dim, "dimension_total_capital": dim_total,
                             "portfolio_total_capital": REAL_TOTAL_CAPITAL, "relative_difference": rel_diff})
SEGMENT_SUMS_MATCH_PORTFOLIO = all(r["relative_difference"] < 1e-9 for r in _sum_check_rows)
print(f"[CROSS-CHECK] Every real dimension's segment capital sums back to the real portfolio total "
      f"(${REAL_TOTAL_CAPITAL:,.0f}): {'EXACT MATCH (all dimensions)' if SEGMENT_SUMS_MATCH_PORTFOLIO else 'MISMATCH FOUND'}.")

_hhi_bounds_rows = []
for _, r in hhi_df.iterrows():
    lower_bound = 1.0 / r["n_segments"] if r["n_segments"] > 0 else 0.0
    within_bounds = (lower_bound - 1e-9) <= r["hhi_fraction"] <= (1.0 + 1e-9)
    _hhi_bounds_rows.append({"dimension": r["dimension"], "hhi_fraction": r["hhi_fraction"],
                              "lower_bound_1_over_n": lower_bound, "within_bounds": bool(within_bounds)})
HHI_WITHIN_MATHEMATICAL_BOUNDS = all(r["within_bounds"] for r in _hhi_bounds_rows)
print(f"[CROSS-CHECK] HHI is within its mathematical bounds [1/N, 1] for "
      f"{'every' if HHI_WITHIN_MATHEMATICAL_BOUNDS else 'NOT every'} real dimension.")

# ---------------------------------------------------------------------------
# SECTION 7 — CONCENTRATION VALIDATION VERDICT (this notebook's equivalent
# of the suite's two-tier verdict -- deliberately NOT named "Statistical
# Robustness Verdict": HHI is a deterministic concentration measure, not a
# statistical test against a real TARGET, so this tier validates the
# computation is mathematically correct and internally consistent, stated
# explicitly rather than forcing an ill-fitting statistical test here.)
# ---------------------------------------------------------------------------
validation_checks = [
    ("segment_capital_sums_match_portfolio_total", SEGMENT_SUMS_MATCH_PORTFOLIO),
    ("hhi_within_mathematical_bounds", HHI_WITHIN_MATHEMATICAL_BOUNDS),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "CONCENTRATION METRICS VALIDATED — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET VALIDATED — failed: " + ", ".join(_failed_validation_checks) +
         " (this notebook's concentration-metrics-consistency gate, distinct from a "
         "statistical-significance test -- HHI is a deterministic concentration measure, not "
         "tested against a real TARGET)"
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Concentration validation verdict: {ANALYSIS_VERDICT}")

_most_concentrated = hhi_df.sort_values("hhi_points", ascending=False).iloc[0]
_least_concentrated = hhi_df.sort_values("hhi_points", ascending=True).iloc[0]

# ---------------------------------------------------------------------------
# SECTION 8 — Inline charts
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_colors = _palette(len(hhi_df))
axes[0].bar(hhi_df["dimension"], hhi_df["hhi_points"], color=_colors)
axes[0].axhline(DOJ_FTC_UNCONCENTRATED_THRESHOLD, color="gray", linestyle="--", linewidth=1,
                 label=f"Unconcentrated < {DOJ_FTC_UNCONCENTRATED_THRESHOLD:.0f}")
axes[0].axhline(DOJ_FTC_HIGHLY_CONCENTRATED_THRESHOLD, color="crimson", linestyle="--", linewidth=1,
                 label=f"Highly Concentrated > {DOJ_FTC_HIGHLY_CONCENTRATED_THRESHOLD:.0f}")
axes[0].set_ylabel("HHI (0-10,000-point scale)"); axes[0].set_title("Real Capital Concentration (HHI) by Dimension")
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right"); axes[0].legend(fontsize=8)
_top_seg_dim = _most_concentrated["dimension"]
_top_seg_table = segment_tables[_top_seg_dim].head(8)
axes[1].bar(_top_seg_table["segment_value"].astype(str), _top_seg_table["capital_share"], color=_colors)
axes[1].set_ylabel("Share of Real Capital"); axes[1].set_title(f"Real Capital Share by Segment — {_top_seg_dim}")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_05_capital_concentration.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 9 — Pipeline Integrity Checks (structural)
# ---------------------------------------------------------------------------
checks = [
    ("real_data_loaded", N_SCOPE > 0),
    ("required_columns_present", len(missing_cols) == 0 and len(missing_dims) == 0),
    ("hhi_finite_all_dimensions", bool(np.isfinite(hhi_df["hhi_points"].to_numpy()).all())),
    ("capital_shares_sum_to_one_per_dimension", bool(all(
        abs(segment_tables[d]["capital_share"].sum() - 1.0) < 1e-9 for d in ALL_DIMENSIONS
    ))),
    ("cpu_thread_ceiling_applied_before_import", os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Pipeline integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 10 — Reporting & Packaging (SOP Stage 5)
# ---------------------------------------------------------------------------
csv_paths = write_csv_outputs(
    {"notebook_05_hhi_by_dimension": hhi_df, "notebook_05_segment_capital_shares": all_segments_df},
    REPORTS_DIR,
)

ASSUMPTIONS = {
    "HHI definition": "Sum of squared real capital shares per segment, x10,000 (conventional points scale)",
    "Interpretive bands (borrowed convention)": f"Unconcentrated < {DOJ_FTC_UNCONCENTRATED_THRESHOLD:.0f}; "
        f"Moderately Concentrated {DOJ_FTC_UNCONCENTRATED_THRESHOLD:.0f}-{DOJ_FTC_HIGHLY_CONCENTRATED_THRESHOLD:.0f}; "
        f"Highly Concentrated > {DOJ_FTC_HIGHLY_CONCENTRATED_THRESHOLD:.0f}",
}
ASSUMPTION_NOTES = {
    "HHI definition": "Standard concentration metric; here applied to real Basel capital requirement "
                       "shares rather than market shares -- Notebook 01's Pillar-1 closed form assumes "
                       "infinite granularity (zero name-concentration capital charge) [BCBS05], so this "
                       "is a genuine Pillar-2-style addition, not a duplicate of Notebook 01.",
    "Interpretive bands (borrowed convention)": "U.S. DOJ/FTC Horizontal Merger Guidelines HHI bands, "
        "widely borrowed in credit-portfolio concentration-risk practice -- a disclosed interpretive "
        "convention, not a Basel-mandated portfolio threshold.",
}

STORY_HHI_CHART = [
    f"Real capital concentration is highest by {_most_concentrated['dimension']} "
    f"(HHI={_most_concentrated['hhi_points']:.0f}, {_most_concentrated['interpretation']}) and lowest by "
    f"{_least_concentrated['dimension']} (HHI={_least_concentrated['hhi_points']:.0f}, "
    f"{_least_concentrated['interpretation']}).",
    f"Concentration validation verdict: {ANALYSIS_VERDICT}.",
]
STORY_SEGMENT_CHART = [
    f"'{_most_concentrated['largest_segment']}' alone accounts for "
    f"{_most_concentrated['largest_segment_share']:.1%} of real capital within {_top_seg_dim} -- the "
    f"single largest real concentration exposure found across every dimension examined.",
]
INSIGHTS = [{
    "headline": f"Real capital is {_most_concentrated['interpretation'].lower()} by {_most_concentrated['dimension']}",
    "specific": f"HHI={_most_concentrated['hhi_points']:.0f} across {int(_most_concentrated['n_segments'])} real "
                f"segments (effective N={_most_concentrated['effective_n_segments']:.2f}); largest segment "
                f"'{_most_concentrated['largest_segment']}' holds {_most_concentrated['largest_segment_share']:.1%} "
                f"of real capital.",
    "measurable": f"HHI computed for all {len(ALL_DIMENSIONS)} real dimensions available in this dataset in "
                  f"{HHI_RUNTIME_S}s.",
    "achievable": "No further tuning required this cycle." if ANALYSIS_ROBUST else
                  "Investigate the failing concentration-metrics check(s) above before trusting these figures.",
    "relevant": "Fills the concentration-risk gap Notebook 01's Pillar-1 closed form deliberately does not "
                "price (infinite-granularity assumption) -- a real Pillar-2-style view for risk management.",
    "timebound": "Re-run after any Notebook 01 update or a new real segment dimension becomes available.",
}]

word_path = build_word_report(
    REPORTS_DIR / "notebook_05_report.docx",
    title="Mega Project 2 — Notebook 05: Capital Concentration by Segment",
    subtitle="Real Herfindahl-Hirschman Index (HHI) across every real segment/geography dimension",
    exec_summary=[
        f"{N_SCOPE:,} real applicants (Notebook 01's real capital output, reused not recomputed).",
        f"{len(ALL_DIMENSIONS)} real dimensions examined: {', '.join(ALL_DIMENSIONS)}.",
        f"Most concentrated: {_most_concentrated['dimension']} (HHI={_most_concentrated['hhi_points']:.0f}, "
        f"{_most_concentrated['interpretation']}).",
        f"Concentration validation verdict: {ANALYSIS_VERDICT}",
    ],
    sections=[
        {"heading": "HHI by Real Dimension",
         "paragraphs": ["Real Herfindahl-Hirschman Index, effective segment count, and largest-segment "
                        "share per real dimension."],
         "table": {"headers": ["Dimension", "N Segments", "HHI", "Effective N", "Interpretation", "Largest Segment", "Share"],
                   "rows": [[r["dimension"], int(r["n_segments"]), f"{r['hhi_points']:.0f}",
                             f"{r['effective_n_segments']:.2f}", r["interpretation"], r["largest_segment"],
                             f"{r['largest_segment_share']:.1%}"] for _, r in hhi_df.iterrows()]},
         "image_path": ARTIFACTS_DIR / "notebook_05_capital_concentration.png", "story": STORY_HHI_CHART},
        {"heading": f"Real Capital Share by Segment — {_top_seg_dim} (most concentrated dimension)",
         "paragraphs": ["Real capital share per segment within the most concentrated real dimension."],
         "table": {"headers": ["Segment", "N Applicants", "Total Capital", "Capital Share"],
                   "rows": [[r["segment_value"], f"{int(r['n_applicants']):,}", f"${r['total_capital']:,.0f}",
                             f"{r['capital_share']:.1%}"] for _, r in _top_seg_table.iterrows()]},
         "story": STORY_SEGMENT_CHART},
    ],
    insights=INSIGHTS,
)

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_05_workbook.xlsx",
    assumptions=ASSUMPTIONS, assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "HHI by Dimension", "headers": list(hhi_df.columns),
         "rows": hhi_df.astype(object).values.tolist(), "highlight_col": "hhi_points"},
        {"name": "Segment Capital Shares", "headers": list(all_segments_df.columns),
         "rows": all_segments_df.astype(object).values.tolist(), "highlight_col": "capital_share"},
    ],
    formula_sheet={"name": "Portfolio Summary",
                   "rows": [("Real Total Capital", REAL_TOTAL_CAPITAL),
                            ("Most concentrated dimension", _most_concentrated["dimension"]),
                            ("Most concentrated HHI", _most_concentrated["hhi_points"]),
                            ("Least concentrated dimension", _least_concentrated["dimension"]),
                            ("Least concentrated HHI", _least_concentrated["hhi_points"])]},
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_05_dashboard.html",
    title="Mega Project 2 — Capital Concentration by Segment",
    subtitle=f"{N_SCOPE:,} real applicants — HHI across {len(ALL_DIMENSIONS)} real dimensions",
    kpi_cards=[
        {"label": "Real Applicants", "value": f"{N_SCOPE:,}"},
        {"label": "Real Total Capital", "value": f"${REAL_TOTAL_CAPITAL:,.0f}"},
        {"label": "Most Concentrated Dimension", "value": _most_concentrated["dimension"]},
        {"label": "Highest HHI", "value": f"{_most_concentrated['hhi_points']:.0f} ({_most_concentrated['interpretation']})"},
    ],
    charts=[
        {"id": "hhiByDimension", "title": "Real Capital Concentration (HHI) by Dimension", "type": "bar",
         "labels": hhi_df["dimension"].tolist(),
         "datasets": [{"label": "HHI (points)", "data": hhi_df["hhi_points"].round(0).tolist()}],
         "story": STORY_HHI_CHART},
    ],
    insights=INSIGHTS,
    data_table={"title": "Segment Capital Shares (all dimensions)", "columns": list(all_segments_df.columns),
                "rows": all_segments_df.values.tolist(), "filter_column": "dimension"},
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s).")

# ---------------------------------------------------------------------------
# SECTION 11 — Save artifacts + governance stamp (idempotent)
# ---------------------------------------------------------------------------
summary = {
    "notebook": "05_capital_concentration_by_segment",
    "mega_project": "Mega Project 2 - Regulatory Capital & Expected Loss",
    "problem": "Problem 5 - Capital Concentration by Segment",
    "random_seed": SEED,
    "n_applicants": N_SCOPE,
    "upstream_dependency": {"source_notebook": "Mega Project 2 / Notebook 01",
                             "reused_not_recomputed": True, "columns_reused": required_cols},
    "dimensions_examined": ALL_DIMENSIONS,
    "hhi_by_dimension": hhi_df.to_dict(orient="records"),
    "segment_capital_shares": all_segments_df.to_dict(orient="records"),
    "hhi_runtime_seconds": HHI_RUNTIME_S,
    "cross_checks": {
        "segment_capital_sums_match_portfolio_total": {
            "holds": bool(SEGMENT_SUMS_MATCH_PORTFOLIO), "detail": _sum_check_rows,
        },
        "hhi_within_mathematical_bounds": {
            "holds": bool(HHI_WITHIN_MATHEMATICAL_BOUNDS), "detail": _hhi_bounds_rows,
        },
    },
    "concentration_validation": {
        "validation_checks": {name: bool(ok) for name, ok in validation_checks},
        "failed_validation_checks": _failed_validation_checks,
        "deployment_verdict": ANALYSIS_VERDICT,
        "note": "This notebook validates concentration-metric MECHANICS (mathematical bounds, sum-to-"
                "total identity), not statistical significance against a real TARGET -- HHI is a "
                "deterministic concentration measure (see module docstring).",
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": ["notebook_05_report.docx", "notebook_05_workbook.xlsx", "notebook_05_dashboard.html"]
                           + [f"{stem}.csv" for stem in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_05_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[DONE] Mega Project 2 / Notebook 05 complete in {summary['runtime_seconds']}s. "
      f"Most concentrated real dimension: {_most_concentrated['dimension']} "
      f"(HHI={_most_concentrated['hhi_points']:.0f}, {_most_concentrated['interpretation']}). "
      f"Concentration validation verdict: {ANALYSIS_VERDICT}.")
